# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides an example workflow for loading, examining, and processing the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

Use this notebook as a template for analyzing other `mlcroissant` datasets.

In [ ]:
# Ensure `mlcroissant` is installed in your environment
!pip install mlcroissant

## 1. Data Loading

Load and inspect the Croissant metadata and records from the dataset.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Show dataset overview
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Version: {metadata.version}")

## 2. Data Overview

List record sets, fields, and their `@id`s defined in the schema.

In [ ]:
# List all record sets with their @id and name
print("Available record sets:")
for record_set in metadata.record_sets:
    print(f"  - @id: {record_set.id}   name: {getattr(record_set, 'name', '(no name)')}")
    if hasattr(record_set, 'fields'):
        print("    Fields and their @id:")
        for field in record_set.fields:
            print(f"      - @id: {field.id}   name: {getattr(field, 'name', '(no name)')}")

## 3. Data Extraction

Load data from each record set into a DataFrame for analysis. Record set and field `@id`s are used to reference the data consistently.

In [ ]:
# Collect all record set @ids
record_sets = [rs.id for rs in metadata.record_sets]

dataframes = {}
# Load each record set as a pandas DataFrame
for record_set_id in record_sets:
    print(f"Loading records from: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Loaded {df.shape[0]} rows and {df.shape[1]} columns.")

# Select first record set for demonstration; replace with desired @id as needed
if record_sets:
    example_record_set_id = record_sets[0]
    example_df = dataframes[example_record_set_id]
    print(f"\nFields for record set {example_record_set_id}:")
    print(example_df.columns.tolist())
    example_df.head()
else:
    print("No record sets found in the dataset.")

## 4. Exploratory Data Analysis (EDA)

Process, filter, and transform the data using field `@id`s for references. This example demonstrates:
- Filtering by numeric fields
- Normalizing a numeric column
- Grouping by a categorical column

Update the variable assignments with the appropriate `@id`s from your selected record set.

In [ ]:
# Example: select a numeric field and a group field by @id from the previous list
# Update these variables using the print-out of available fields (@id values!)

# Set up example field @ids (replace with real ones as per Section 2 output)
numeric_field_id = None  # e.g., 'http://mlcommons.org/croissant/field/age'
group_field_id = None    # e.g., 'http://mlcommons.org/croissant/field/sex'

# If the dataframe contains any numeric columns, pick one automatically for this demo
import numpy as np

example_numeric_field = None
example_group_field = None
if example_df.shape[1] > 0:
    for col in example_df.columns:
        if np.issubdtype(example_df[col].dtype, np.number):
            example_numeric_field = col
            break
    # Pick the first non-numeric field for grouping
    for col in example_df.columns:
        if not np.issubdtype(example_df[col].dtype, np.number):
            example_group_field = col
            break
    if example_numeric_field:
        numeric_field_id = example_numeric_field
    if example_group_field:
        group_field_id = example_group_field
    
print(f"Using numeric field @id: {numeric_field_id}")
print(f"Using group field @id: {group_field_id}")

# Proceed only if a numeric field exists
if numeric_field_id is not None and numeric_field_id in example_df.columns:
    threshold = 10
    filtered_df = example_df[example_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())
    
    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Group by group_field
    if group_field_id is not None and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization

Visualize data distributions or relationships between fields. Adjust the field `@id`s and labels as appropriate for your analysis.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Simple visualization: histogram of numeric field
if numeric_field_id is not None and numeric_field_id in example_df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(example_df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

# If both numeric and group fields, plot boxplot
if numeric_field_id is not None and group_field_id is not None and \
   numeric_field_id in example_df.columns and group_field_id in example_df.columns:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=example_df[group_field_id], y=example_df[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion

With the `mlcroissant` library, we've loaded and explored the FAIR^2 dataset, examining its record sets, fields, and values. We demonstrated filtering, normalization, grouping, and produced basic visualizations with reference to Croissant `@id`s. This workflow serves as a template for working with modern, schema-driven research datasets.